In [1]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np
from shapely.geometry import box


In [2]:
tif_path = r"D:\wenqu\mosaic\rgb\site2a_rgb.tif"          # the raster you want to match (georeferencing & shape)
shp_path = r"D:\wenqu\mosaic\NNDVI\shadow_shapefile\site2a_shadow.shp" # digitized polygons with a 'class' field
out_pred  = r"D:\wenqu\mosaic\NNDVI\mask\site2a_mask_test.tif"



In [3]:
# open image
with rasterio.open(tif_path) as src:
    img = src.read()                  # (bands, H, W)
    profile = src.profile.copy()
    transform, crs = src.transform, src.crs
    nodata = src.nodata
    bands, H, W = img.shape

# read polygons and match CRS
gdf = gpd.read_file(shp_path).to_crs(crs)

# make a label raster from Id
label = rasterize(
    [(geom, int(v)) for geom, v in zip(gdf.geometry, gdf["Id"])],
    out_shape=(H, W),
    transform=transform,
    fill=255,     # unlabeled pixels
    all_touched=False,
    dtype="uint8"
)

# build training set: where label is 0 or 1
X = img.reshape(bands, -1).T          # (Npix, bands)
y = label.reshape(-1)                 # (Npix,)

train_mask = (y == 0) | (y == 1)

# drop nodata / NaNs
if nodata is not None:
    nod = np.any(img == nodata, axis=0).reshape(-1)
    train_mask &= ~nod
nan_mask = np.any(~np.isfinite(X), axis=1)
train_mask &= ~nan_mask

X_train = X[train_mask]
y_train = y[train_mask]

print(f"Training samples: {X_train.shape[0]:,}  |  Bands: {X_train.shape[1]}")

# train RF
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

# predict full scene (for very large rasters, do this in tiles)
y_pred = rf.predict(X)
pred = y_pred.reshape(H, W).astype("uint8")

# set invalid pixels to 255 (nodata) if desired
if nodata is not None:
    invalid = np.any(img == nodata, axis=0)
    pred[invalid] = 255

profile.update(count=1, dtype="uint8", nodata=255, compress="lzw")
with rasterio.open(out_pred, "w", **profile) as dst:
    dst.write(pred, 1)

print("Saved:", out_pred)


Training samples: 2,279  |  Bands: 3
Saved: D:\wenqu\mosaic\NNDVI\mask\site2a_mask_test.tif


In [1]:
import geopandas as gpd
import numpy as np

gdf = gpd.read_file(r"D:\wenqu\mosaic\NNDVI\shadow_shapefile\site2c_shadow2.shp")

print("Row count:", len(gdf))
print("gdf.empty:", gdf.empty)  # True => 0 rows

# Are all geometries missing or empty?
all_geom_missing = gdf.geometry.isna().all()
all_geom_empty   = gdf.geometry.is_empty.all() if not gdf.empty else True
print("All geometries None/NaN:", all_geom_missing)
print("All geometries empty:", all_geom_empty)

# Bounds check (will be NaNs if geometries are all empty/missing)
tb = getattr(gdf, "total_bounds", None)
print("total_bounds:", tb)
bounds_nan = tb is None or np.isnan(tb).any()

if gdf.empty:
    print("➡️ Shapefile has no rows (empty).")
elif all_geom_missing or all_geom_empty or bounds_nan:
    print("➡️ Shapefile has rows but geometries are empty/invalid.")
else:
    print(f"✅ Shapefile has {len(gdf)} valid row(s) with geometry.")


PROJ: proj_create_from_database: Cannot find proj.db


Row count: 0
gdf.empty: True
All geometries None/NaN: True
All geometries empty: True
total_bounds: [nan nan nan nan]
➡️ Shapefile has no rows (empty).


In [1]:
import geopandas as gpd
gdf = gpd.read_file(r"D:\wenqu\mosaic\NNDVI\shadow_shapefile\site2c_shadow.shp")
print(len(gdf))
print(gdf.head())


PROJ: proj_create_from_database: Cannot find proj.db


42
   Id                                           geometry
0   0  POLYGON ((-150.12665 67.35894, -150.12665 67.3...
1   0  POLYGON ((-150.12669 67.35895, -150.12669 67.3...
2   0  POLYGON ((-150.12663 67.35898, -150.12662 67.3...
3   0  POLYGON ((-150.12667 67.35904, -150.12667 67.3...
4   0  POLYGON ((-150.12619 67.35891, -150.12619 67.3...


In [4]:
import fiona
with fiona.open(r"D:\wenqu\mosaic\NNDVI\shadow_shapefile\site2c_shadow2.shp") as src:
    print("Feature count:", len(src))


Feature count: 0
